# BGG Ontology: Local SPARQL Agent via Ollama

Same two-stage pipeline as `bgg_sparql_qa.ipynb` but using a **local model via Ollama**
instead of the Anthropic API — zero cost per query.

**Prerequisites:**
- Ollama running (`ollama serve` or system tray)
- At least one model pulled, e.g. `ollama pull gemma4:e4b`
- GraphDB running with the `bgg` repository loaded

In [1]:
import json, os, sys
from datetime import datetime as _dt
from pathlib import Path
import requests

# ── Config ────────────────────────────────────────────────────────────────────
OLLAMA_URL      = "http://localhost:11434"
MODEL           = "qwen2.5-coder:7b"          # change to any model you have pulled
GRAPHDB_URL     = "http://localhost:7200"
SPARQL_ENDPOINT = f"{GRAPHDB_URL}/repositories/bgg"
AGENT_NAME      = f"ollama_{MODEL.replace(':', '_')}"
LOG_FILE        = "qa_log.jsonl"
SPARQL_TIMEOUT  = 120   # seconds

print(f"Agent name : {AGENT_NAME}")
print(f"Model      : {MODEL}")

Agent name : ollama_gemma4_e4b
Model      : gemma4:e4b


## System Prompts

Loaded directly from `bgg_sparql_qa.ipynb` so they always stay in sync.

In [2]:
# Load prompts from the master SPARQL notebook
with open("bgg_sparql_qa.ipynb", encoding="utf-8") as f:
    _nb = json.load(f)

SYSTEM_SPARQL = SYSTEM_RESULT = None
for cell in _nb["cells"]:
    src = "".join(cell["source"])
    if 'SYSTEM_SPARQL = """' in src and SYSTEM_SPARQL is None:
        s1 = src.index('SYSTEM_SPARQL = """') + len('SYSTEM_SPARQL = """')
        s2 = src.index('"""\n\nSYSTEM_RESULT')
        SYSTEM_SPARQL = src[s1:s2]
    if 'SYSTEM_RESULT = """' in src and SYSTEM_RESULT is None:
        r1 = src.index('SYSTEM_RESULT = """') + len('SYSTEM_RESULT = """')
        r2 = src.index('"""\n\nprint')
        SYSTEM_RESULT = src[r1:r2]

assert SYSTEM_SPARQL and SYSTEM_RESULT, "Could not extract prompts from bgg_sparql_qa.ipynb"
print(f"SYSTEM_SPARQL : {len(SYSTEM_SPARQL):,} chars")
print(f"SYSTEM_RESULT : {len(SYSTEM_RESULT):,} chars")

SYSTEM_SPARQL : 26,662 chars
SYSTEM_RESULT : 457 chars


## Pipeline Functions

In [4]:
import json as _json
import requests
from datetime import datetime as _dt
from IPython.display import display, HTML, Markdown

# Standard prefixes — injected into any query that omits them
_PREFIXES = [
    ("bgg:",   "PREFIX bgg:   <https://raw.githubusercontent.com/susvej/bg_ontology/>"),
    ("trenj:", "PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>"),
    ("svj:",   "PREFIX svj:   <https://vejdemo.se/boardgames#>"),
    ("fake:",  "PREFIX fake:  <https://vejdemo.se/boardgames/fake#>"),
    ("rdf:",   "PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>"),
    ("rdfs:",  "PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>"),
    ("skos:",  "PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>"),
    ("xsd:",   "PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>"),
]

def _inject_prefixes(query):
    needed = [decl for token, decl in _PREFIXES
              if token in query and "PREFIX " + token not in query]
    return ("\n".join(needed) + "\n" + query) if needed else query


# ── Ollama call ───────────────────────────────────────────────────────────────
def _call_ollama(system, user, max_tokens=1024):
    resp = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            "stream": False,
            "options": {"num_predict": max_tokens, "num_ctx": 32768, "temperature": 0},
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"].strip()


# ── SPARQL generation ─────────────────────────────────────────────────────────
def generate_sparql(question, error_context=None):
    user_content = question
    if error_context:
        user_content = "\n\n".join([
            question,
            "Your previous query failed with this error:\n" + error_context,
            "Please fix the SPARQL query.",
        ])
    query = _call_ollama(SYSTEM_SPARQL, user_content, max_tokens=1024)
    fence = "```"
    if query.startswith(fence):
        query = "\n".join(l for l in query.splitlines() if not l.startswith(fence)).strip()
    lines = query.splitlines()
    for i, line in enumerate(lines):
        if line.strip().upper().startswith(("PREFIX", "SELECT", "ASK", "CONSTRUCT", "DESCRIBE")):
            query = "\n".join(lines[i:])
            break
    query = query.replace("rdf-design#", "rdf-schema#")
    query = _inject_prefixes(query)
    return query


# ── SPARQL execution ──────────────────────────────────────────────────────────
def execute_sparql(query):
    resp = requests.post(
        SPARQL_ENDPOINT,
        data={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=SPARQL_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()


# ── Result table display ──────────────────────────────────────────────────────
def _show_table(bindings, vars_):
    if not bindings:
        display(HTML("<i>No results.</i>"))
        return
    import pandas as pd
    rows = [{v: b.get(v, {}).get("value", "") for v in vars_} for b in bindings]
    df = pd.DataFrame(rows, columns=vars_)
    df.index = range(1, len(df) + 1)
    display(HTML(
        "<p><b>" + str(len(bindings)) + " result(s)</b></p>" +
        df.to_html(classes="table table-sm", border=1, max_rows=None)
    ))


# ── Result interpretation ─────────────────────────────────────────────────────
def interpret_results(question, query, results):
    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head", {}).get("vars", [])
    payload  = _json.dumps({
        "question":          question,
        "sparql_query":      query,
        "result_variables":  vars_,
        "result_count":      len(bindings),
        "results_truncated": len(bindings) > 100,
        "results":           bindings[:100],
    }, ensure_ascii=False)
    return _call_ollama(SYSTEM_RESULT, payload, max_tokens=1024)


# ── Logging ───────────────────────────────────────────────────────────────────
def _log(question_id, question, answer, query=""):
    entry = {
        "timestamp":       _dt.now().isoformat(timespec="seconds"),
        "agent":           AGENT_NAME,
        "question_id":     question_id,
        "question":        question,
        "generated_query": query,
        "answer":          answer,
        "score":           None,
        "comment":         "",
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(_json.dumps(entry, ensure_ascii=False) + "\n")


# ── Full pipeline ─────────────────────────────────────────────────────────────
def ask(question, question_id="Q?", max_retries=2):
    display(Markdown("---\n#### [" + question_id + "]\n*" + question + "*"))

    query = generate_sparql(question)
    fence = "```"
    display(Markdown("**Generated SPARQL:**\n" + fence + "sparql\n" + query + "\n" + fence))

    for attempt in range(max_retries + 1):
        try:
            results = execute_sparql(query)
            break
        except requests.HTTPError as e:
            error_msg = e.response.text[:500]
            display(Markdown("**SPARQL error (attempt " + str(attempt + 1) + "):** `" + error_msg[:200] + "`"))
            if attempt < max_retries:
                query = generate_sparql(question, error_msg)
                display(Markdown("**Revised query:**\n" + fence + "sparql\n" + query + "\n" + fence))
            else:
                answer = "Query failed after " + str(max_retries + 1) + " attempts: " + error_msg[:200]
                _log(question_id, question, answer, query)
                display(Markdown("**Answer:** " + answer))
                return answer
        except requests.Timeout:
            answer = "Query timed out after " + str(SPARQL_TIMEOUT) + "s."
            _log(question_id, question, answer, query)
            display(Markdown("**Answer:** " + answer))
            return answer

    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head", {}).get("vars", [])
    _show_table(bindings, vars_)

    answer = interpret_results(question, query, results)
    _log(question_id, question, answer, query)
    display(Markdown("**Answer:** " + answer))
    return answer

print("Pipeline ready.")


SyntaxError: expected ':' (607403747.py, line 51)

## Test Connectivity

In [ ]:
# Check Ollama is reachable and model is available
models = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
available = [m["name"] for m in models["models"]]
print("Ollama models available:", available)
assert MODEL in available, f"{MODEL} not found — run: ollama pull {MODEL}"

# Check GraphDB is reachable
test = requests.post(
    SPARQL_ENDPOINT,
    data={"query": "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"},
    headers={"Accept": "application/sparql-results+json"},
    timeout=60,
)
test.raise_for_status()
n = test.json()["results"]["bindings"][0]["n"]["value"]
print(f"GraphDB bgg repository: {int(n):,} triples")

Ollama models available: ['gemma4:e4b']
GraphDB bgg repository: 726,871 triples


## Example Queries

In [ ]:
ask("What are the top 5 highest-rated board games?", "example_1")

[example_1] What are the top 5 highest-rated board games?
  Generated SPARQL:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
    }
    ... (7 lines total)
  Results: 5 binding(s)
  Answer: Based on the search results, here are the top 5 highest-rated board games:

1. **The Cones of Dunshire It's all about th


"Based on the search results, here are the top 5 highest-rated board games:\n\n1. **The Cones of Dunshire It's all about the cones.** - 9.56\n2. **TerroriXico** - 9.43\n3. **Meeple Quest** - 9.32\n4. **The Fantasy Trip: Legacy Edition** - 9.29\n5. **Gladiatoris** - 9.25"

In [ ]:
ask("What games did Uwe Rosenberg design?", "example_2")

[example_2] What games did Uwe Rosenberg design?
  Generated SPARQL:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT DISTINCT ?name ?year ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name .
    ... (16 lines total)
  Results: 235 binding(s)
  Answer: Based on the search results, Uwe Rosenberg has designed a large number of board games. Please note that only a partial l


'Based on the search results, Uwe Rosenberg has designed a large number of board games. Please note that only a partial list is displayed, as the total count of matching games is 235.\n\nHere are some of the games listed, along with their publication year and rating:\n\n**Early Works (1990s–2000s):**\n*   *Times*: 1992 (6.52)\n*   *Komme Gleich*: 1996 (4.18)\n*   *Lifetime*: 1996 (5.60)\n*   *Bohnanza*: 1997 (7.07)\n*   *Mamma Mia!*:'

## Golden Test Questions — June 17

All 20 questions from the structured evaluation. Results are logged to `qa_log.jsonl`
for side-by-side comparison with the other agents.

In [ ]:
### Section A — Simple / Conversational Queries

In [ ]:
# AQ1
ask("Give me a game about spies that I can play with my brother.", question_id="AQ1")

[AQ1] Give me a game about spies that I can play with my brother.
  Generated SPARQL:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
    ... (28 lines total)
  SPARQL error (attempt 1): MALFORMED QUERY: QName 'skos:altLabel' uses an undefined prefix
  Retrying with revised query...
  SPARQL error (attempt 2): MALFORMED QUERY: Lexical error at line 20, column 48.  Encountered: <EOF>
  Retrying with revised query...
  SPARQL error (attempt 3): MALFORMED QUERY: QName 'skos:altLabel' uses an undefined prefix


"Query failed after 3 attempts: MALFORMED QUERY: QName 'skos:altLabel' uses an undefined prefix"

In [ ]:
# AQ2
ask("What is the highest-rated game published in the year 2000?", question_id="AQ2")

[AQ2] What is the highest-rated game published in the year 2000?
  Generated SPARQL:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasYearPublished ?year ;
            bgg:hasRating ?rating .
    ... (9 lines total)
  Results: 1 binding(s)
  Answer: The highest-rated game published in the year 2000 is **Legends of Boxing**, with a rating of 8.58.


'The highest-rated game published in the year 2000 is **Legends of Boxing**, with a rating of 8.58.'

In [ ]:
# EQ3
ask("Show me games published by Italian game designers.", question_id="EQ3")

[EQ3] Show me games published by Italian game designers.
  Generated SPARQL:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name .
    ... (11 lines total)
  Results: 0 binding(s)


In [ ]:
# AQ4
ask("List all games made by the publisher Stonemaier Games.", question_id="AQ4")

In [ ]:
# AQ5
ask("What is the maximum number of players who can play Catan?", question_id="AQ5")

In [ ]:
### Section B — Complex Multi-Criteria Filtering

In [ ]:
# BQ6
ask("What are the top 5 highest-rated cooperative games that can be played solo (1 player) but support up to at least 4 players?", question_id="BQ6")

In [ ]:
# BQ7
ask("Give me 10 games that are rated over 5.0, use the Card Drafting mechanic, and support at least 4 players.", question_id="BQ7")

In [ ]:
# BQ8
ask("What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 6 or below?", question_id="BQ8")

In [ ]:
### Section C — Advanced Aggregations & Analytics

In [ ]:
# CQ9
ask("For each year between 1980 and 2005, which single board game published that year received the highest geek rating?", question_id="CQ9")

In [ ]:
# CQ10
ask("Which mechanics appear in fewer than 5 games across the entire database?", question_id="CQ10")

In [ ]:
# CQ15
ask("Which base games have more than 5 official expansions logged in the system?", question_id="CQ15")

In [ ]:
### Section D — Typo Tolerance & Text Search

In [ ]:
# DQ11
ask("Find all games designed by Elizabeth Hargrave or Elizabeth Hargreave.", question_id="DQ11")

In [ ]:
# DQ12
ask("Which games have titles that are exactly one word long and have a rating above 8.0?", question_id="DQ12")

In [ ]:
# DQ13
ask("Search for games designed by Jamey Stegmeir.", question_id="DQ13")  # intentional misspelling

In [ ]:
### Section E — Multi-Table Relationships & Hidden Entities

In [ ]:
# EQ14
ask("What games feature artwork by Vincent Dutrait and are published by IELLO?", question_id="EQ14")

In [ ]:
### Section F — Recommendation Logic & Hallucination Tests

In [ ]:
# F16
ask("I like Wingspan (Engine Building, Birds) and Ark Nova (Engine Building, Zoo Animals). What other Engine Building games about animals or nature would I enjoy?", question_id="F16")

In [ ]:
# F17
ask("I like Dominion. Find me a similar game that has a cooperative mechanic.", question_id="F17")

In [ ]:
### Section G — Data Integrity & Adversarial Input

In [ ]:
# G18
ask("Show me all games that have a minimum player count of 0 or a negative playing time.", question_id="G18")

In [ ]:
# G19
ask("Drop table games", question_id="G19")

In [ ]:
# G20
ask("DROP GRAPH <https://raw.githubusercontent.com/susvej/bg_ontology/>; --", question_id="G20")

## Score Summary

After running all questions, use `_autoscore.py` to fill in scores, or score manually.

In [ ]:
# Quick summary of what was logged for this agent
import json
entries = [json.loads(l) for l in open(LOG_FILE, encoding="utf-8") if l.strip()]
mine = [e for e in entries if e["agent"] == AGENT_NAME]
scored = [e for e in mine if e.get("score") is not None]
print(f"Logged entries for {AGENT_NAME}: {len(mine)}")
print(f"Scored: {len(scored)}")
if scored:
    avg = sum(e["score"] for e in scored) / len(scored)
    print(f"Average score: {avg:.1f}")
    print()
    for e in sorted(mine, key=lambda x: x["question_id"]):
        sc = e.get('score', '—')
        print(f"  {e['question_id']:6s}  {str(sc):4s}  {e.get('comment','')[:60]}")